In [3]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from rank_bm25 import BM25Okapi


# ==================================================
# 1. Load Chroma
# ==================================================

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = Chroma(
    collection_name="langchain",
    persist_directory="chroma_db",
    embedding_function=embedding_model,
)


e:\GenAI\Langchain-Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys

print(sys.executable)

e:\GenAI\Langchain-Learning\.venv\Scripts\python.exe


In [4]:
# ==================================================
# 2. Get all documents
# ==================================================

data = vector_store.get(
    include=["documents", "metadatas"]
)

In [5]:
type(data)

dict

In [7]:
data.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [8]:
type(data["documents"])

list

In [9]:
data["documents"][0]

'AI EVALUATION\nFRAMEWORKS, BENCHMARKS & SAFETY ASSESSMENTS\nTechnical Report\nFocus: LLM & Frontier Models\nStandard: ISO/IEC IEEE 24029\nExecutive Overview: Robust evaluation of Artiﬁcial Intelligence systems has transitioned from an academic\nexercise to a critical requirement for safety, deployment governance, and capability tracking. As systems evolve\ntoward multi-modal reasoning and autonomous agentics, evaluation methodologies must balance static\nstandardized benchmarks with dynamic, adversarial, and human-aligned metrics.\n1. CORE EVALUATION METHODOLOGIES\nAutomated Metrics Deterministic\nStatistical lexical matching (e.g., ROUGE, BLEU, Exact\nMatch) and semantic embedding similarity (BERTScore).\nHigh scalability and low cost, but severely limited in\nmeasuring high-level reasoning, nuance, and creativity.\nHuman Evaluation Gold Standard\nExpert or crowd-sourced human raters assessing\nhelpfulness, accuracy, and style. Extremely accurate for'

In [10]:
documents = [
    Document(
        page_content=text,
        metadata=metadata
    )
    for text, metadata in zip(
        data["documents"],
        data["metadatas"]
    )
]

In [12]:
type(documents)

list

In [37]:
type(documents[0])

langchain_core.documents.base.Document

In [13]:
documents[1]

Document(metadata={'creationdate': '2026-09-13T17:27:02+00:00', 'page_label': '1', 'page': 0, 'creator': 'Chromium', 'total_pages': 2, 'producer': 'Skia/PDF m136', 'source': 'data\\AI Evaluation Report.pdf', 'title': 'AI Evaluation & Benchmarking Report', 'moddate': '2026-09-13T17:27:02+00:00'}, page_content='helpfulness, accuracy, and style. Extremely accurate for\nintent alignment, but suffers from high cost, poor scalability,\nand subjective annotator variance.\nLLM-as-a-Judge Scalable AI\nLeveraging frontier models (e.g., GPT-4) to evaluate outputs\nof target models using structured rubrics or pairwise\ncomparisons. Enables rapid, nuanced scoring but inherits\nself-enhancement and position biases.\nTask-Driven Evals Domain Speciﬁc\nEnd-to-end execution of code, mathematical proofs, or API\ncalling in sandboxed environments. Veriﬁcation relies on\ndeterministic unit test passing rather than language\ngeneration quality.\n2. STANDARD BENCHMARK ECOSYSTEM\nStandardized benchmarks provi

In [14]:
# ==================================================
# 3. Query
# ==================================================

query = (
    "What are the main evaluation metrics "
    "used for classification models?"
)


# ==================================================
# 4. VECTOR SEARCH
# ==================================================

vector_results = vector_store.similarity_search_with_score(
    query,
    k=8
)

In [15]:
type(vector_results)

list

In [19]:
type(vector_results[0])

tuple

In [16]:
vector_results[0]

(Document(id='62964cbe-f067-4f1e-906c-799e8e0f2385', metadata={'producer': 'Skia/PDF m136', 'source': 'data\\Machine Learning & Deep Learning Algorithms and Evaluation Metrics.pdf', 'page_label': '3', 'moddate': '2026-09-13T17:32:39+00:00', 'total_pages': 3, 'creationdate': '2026-09-13T17:32:39+00:00', 'title': 'Machine Learning & Deep Learning Algorithms & Evaluation Metrics', 'page': 2, 'creator': 'Chromium'}, page_content="ML & DL METRICS\nPART III: COMPREHENSIVE MODEL EVALUATION METRICS\nTechnical Reference Compendium\nScope: Quantitative Metrics\nFormat: 3-Page Reference Guide\nEvaluation metrics provide quantitative measures of a model's operational performance. Selecting metrics strictly aligned\nwith problem domain properties (such as class imbalance, extreme outliers, or ranking position preference) is critical to\nprevent misleading performance assessments.\n1. CLASSIFICATION & REGRESSION METRICS\nMetric Name Mathematical Formulation Primary Use Case & Nuance\nAccuracy (TP + 

In [18]:
len(vector_results)

8

In [17]:
print("\n========== VECTOR SEARCH ==========\n")

for rank, (document, score) in enumerate(
    vector_results,
    start=1
):

    print(
        f"Rank {rank} | "
        f"Score {score:.4f} | "
        f"Page {document.metadata.get('page_label')}"
    )



========== VECTOR SEARCH ==========

Rank 1 | Score 0.8107 | Page 3
Rank 2 | Score 0.9646 | Page 3
Rank 3 | Score 0.9923 | Page 3
Rank 4 | Score 1.0851 | Page 1
Rank 5 | Score 1.1188 | Page 1
Rank 6 | Score 1.1790 | Page 1
Rank 7 | Score 1.1799 | Page 1
Rank 8 | Score 1.2209 | Page 1


In [21]:
ls=[("hello",100),("chalo",200),("bolo",300)]
for rank,(document,score) in enumerate(ls,start=1):
    print(f"rank:{rank} document :{document} score:{score}")

rank:1 document :hello score:100
rank:2 document :chalo score:200
rank:3 document :bolo score:300


In [22]:
# ==================================================
# 5. BM25 SEARCH
# ==================================================

tokenized_documents = [
    document.page_content.lower().split()
    for document in documents
]


In [23]:
type(tokenized_documents)

list

In [25]:
type(tokenized_documents[0])

list

In [26]:
tokenized_documents[0][:8]

['ai',
 'evaluation',
 'frameworks,',
 'benchmarks',
 '&',
 'safety',
 'assessments',
 'technical']

In [27]:
bm25 = BM25Okapi(tokenized_documents)
type(bm25)

rank_bm25.BM25Okapi

In [28]:
bm25

In [29]:
tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)
type(bm25_scores)

numpy.ndarray

In [32]:
len(bm25_scores)

18

In [34]:
bm25_scores.size

18

In [35]:
bm25_scores.shape

(18,)

In [36]:
bm25_scores

array([2.01369882, 1.6527824 , 0.80148345, 2.15097644, 2.11059992,
       1.53990213, 1.6573924 , 4.17717891, 0.52210843, 1.63161251,
       1.17421069, 0.52848409, 0.        , 0.        , 0.        ,
       3.20718947, 1.24593334, 1.23309836])

In [38]:
bm25_results = sorted(
    zip(documents, bm25_scores),
    key=lambda x: x[1],
    reverse=True
)
type(bm25_results)

list

In [39]:
bm25_results[:2]

[(Document(metadata={'moddate': '2026-09-13T17:32:39+00:00', 'creator': 'Chromium', 'total_pages': 3, 'title': 'Machine Learning & Deep Learning Algorithms & Evaluation Metrics', 'page': 0, 'creationdate': '2026-09-13T17:32:39+00:00', 'page_label': '1', 'producer': 'Skia/PDF m136', 'source': 'data\\Machine Learning & Deep Learning Algorithms and Evaluation Metrics.pdf'}, page_content='ML & DL ALGORITHMS\nPART I: CLASSICAL PARADIGM & LEARNING TAXONOMY\nTechnical Reference Compendium\nScope: Classical ML, DL & Metrics\nFormat: 3-Page Reference Guide\nExecutive Overview: Machine Learning (ML) algorithms construct mathematical models based on sample training data to\nmake predictions or decisions without explicit rule programming. The paradigm spans Supervised Learning (mapping inputs to\nknown targets), Unsupervised Learning (discovering latent structures), and Reinforcement Learning (optimizing sequential\naction policies). Selecting appropriate algorithms and matching evaluation metrics

In [40]:
len(bm25_results)

18

In [41]:
print("\n========== BM25 SEARCH ==========\n")

for rank, (document, score) in enumerate(
    bm25_results[:2],
    start=1
):

    print(
        f"Rank {rank} | "
        f"Score {score:.4f} | "
        f"Page {document.metadata.get('page_label')}"
    )



========== BM25 SEARCH ==========

Rank 1 | Score 4.1772 | Page 1
Rank 2 | Score 3.2072 | Page 3


In [43]:
len(vector_results)

8

In [44]:
vector_results[0]

(Document(id='62964cbe-f067-4f1e-906c-799e8e0f2385', metadata={'producer': 'Skia/PDF m136', 'source': 'data\\Machine Learning & Deep Learning Algorithms and Evaluation Metrics.pdf', 'page_label': '3', 'moddate': '2026-09-13T17:32:39+00:00', 'total_pages': 3, 'creationdate': '2026-09-13T17:32:39+00:00', 'title': 'Machine Learning & Deep Learning Algorithms & Evaluation Metrics', 'page': 2, 'creator': 'Chromium'}, page_content="ML & DL METRICS\nPART III: COMPREHENSIVE MODEL EVALUATION METRICS\nTechnical Reference Compendium\nScope: Quantitative Metrics\nFormat: 3-Page Reference Guide\nEvaluation metrics provide quantitative measures of a model's operational performance. Selecting metrics strictly aligned\nwith problem domain properties (such as class imbalance, extreme outliers, or ranking position preference) is critical to\nprevent misleading performance assessments.\n1. CLASSIFICATION & REGRESSION METRICS\nMetric Name Mathematical Formulation Primary Use Case & Nuance\nAccuracy (TP + 

In [45]:
# ==================================================
# 6. RRF FUNCTION
# ==================================================

def reciprocal_rank_fusion(
    vector_results,
    bm25_results,
    k=60
):

    rrf_scores = {}

    # ----------------------------------------------
    # Vector ranking contribution
    # ----------------------------------------------

    for rank, (document, _) in enumerate(
        vector_results,
        start=1
    ):

        document_id = document.page_content

        rrf_scores[document_id] = (
            rrf_scores.get(document_id, 0)
            + 1 / (k + rank)
        )


    # ----------------------------------------------
    # BM25 ranking contribution
    # ----------------------------------------------

    for rank, (document, _) in enumerate(
        bm25_results[:8],
        start=1
    ):

        document_id = document.page_content

        rrf_scores[document_id] = (
            rrf_scores.get(document_id, 0)
            + 1 / (k + rank)
        )


    return rrf_scores

In [46]:
# ==================================================
# 7. Calculate RRF scores
# ==================================================

rrf_scores = reciprocal_rank_fusion(
    vector_results,
    bm25_results,
    k=60
)

In [47]:
type(rrf_scores)

dict

In [48]:
rrf_scores.items()

dict_items([("ML & DL METRICS\nPART III: COMPREHENSIVE MODEL EVALUATION METRICS\nTechnical Reference Compendium\nScope: Quantitative Metrics\nFormat: 3-Page Reference Guide\nEvaluation metrics provide quantitative measures of a model's operational performance. Selecting metrics strictly aligned\nwith problem domain properties (such as class imbalance, extreme outliers, or ranking position preference) is critical to\nprevent misleading performance assessments.\n1. CLASSIFICATION & REGRESSION METRICS\nMetric Name Mathematical Formulation Primary Use Case & Nuance\nAccuracy (TP + TN) / (TP + TN + FP + FN) Balanced datasets only; highly misleading under severe class\nimbalance.\nPrecision & Recall Prec = TP / (TP + FP) \xa0|\xa0 Rec = TP / (TP +\nFN)\nPrecision minimizes False Positives; Recall minimizes False\nNegatives (e.g., medical).\nF1-Score 2 · (Precision · Recall) / (Precision + Recall) Harmonic mean balancing precision and recall on imbalanced\ndatasets.\nROC-AUC Area under True P

In [49]:
len(rrf_scores)

12

In [50]:
# ==================================================
# 8. Create document lookup
# ==================================================

document_lookup = {
    document.page_content: document
    for document in documents
}
type(document_lookup)

dict

In [51]:
document_lookup.items()

dict_items([('AI EVALUATION\nFRAMEWORKS, BENCHMARKS & SAFETY ASSESSMENTS\nTechnical Report\nFocus: LLM & Frontier Models\nStandard: ISO/IEC IEEE 24029\nExecutive Overview: Robust evaluation of Artiﬁcial Intelligence systems has transitioned from an academic\nexercise to a critical requirement for safety, deployment governance, and capability tracking. As systems evolve\ntoward multi-modal reasoning and autonomous agentics, evaluation methodologies must balance static\nstandardized benchmarks with dynamic, adversarial, and human-aligned metrics.\n1. CORE EVALUATION METHODOLOGIES\nAutomated Metrics Deterministic\nStatistical lexical matching (e.g., ROUGE, BLEU, Exact\nMatch) and semantic embedding similarity (BERTScore).\nHigh scalability and low cost, but severely limited in\nmeasuring high-level reasoning, nuance, and creativity.\nHuman Evaluation Gold Standard\nExpert or crowd-sourced human raters assessing\nhelpfulness, accuracy, and style. Extremely accurate for', Document(metadata=

In [52]:
len(document_lookup)

18

In [53]:
# ==================================================
# 9. Sort by RRF score
# ==================================================

hybrid_results = sorted(
    rrf_scores.items(),
    key=lambda x: x[1],
    reverse=True
)
type(hybrid_results)

list

In [54]:
len(hybrid_results)

12

In [55]:
hybrid_results[1]

('AI EVALUATION\nFRAMEWORKS, BENCHMARKS & SAFETY ASSESSMENTS\nTechnical Report\nFocus: LLM & Frontier Models\nStandard: ISO/IEC IEEE 24029\nExecutive Overview: Robust evaluation of Artiﬁcial Intelligence systems has transitioned from an academic\nexercise to a critical requirement for safety, deployment governance, and capability tracking. As systems evolve\ntoward multi-modal reasoning and autonomous agentics, evaluation methodologies must balance static\nstandardized benchmarks with dynamic, adversarial, and human-aligned metrics.\n1. CORE EVALUATION METHODOLOGIES\nAutomated Metrics Deterministic\nStatistical lexical matching (e.g., ROUGE, BLEU, Exact\nMatch) and semantic embedding similarity (BERTScore).\nHigh scalability and low cost, but severely limited in\nmeasuring high-level reasoning, nuance, and creativity.\nHuman Evaluation Gold Standard\nExpert or crowd-sourced human raters assessing\nhelpfulness, accuracy, and style. Extremely accurate for',
 0.03076923076923077)

In [56]:
# ==================================================
# 10. Display Hybrid Results
# ==================================================

print("\n========== HYBRID SEARCH (RRF) ==========\n")

for rank, (document_id, score) in enumerate(
    hybrid_results[:8],
    start=1
):

    document = document_lookup[document_id]

    print(f"Rank {rank}")
    print(f"RRF score : {score:.6f}")
    print(
        f"Source    : "
        f"{document.metadata.get('source')}"
    )
    print(
        f"Page      : "
        f"{document.metadata.get('page_label')}"
    )
    print(
        f"Content   : "
        f"{document.page_content[:250]}"
    )
    print("-" * 80)


========== HYBRID SEARCH (RRF) ==========

Rank 1
RRF score : 0.032522
Source    : data\Machine Learning & Deep Learning Algorithms and Evaluation Metrics.pdf
Page      : 3
Content   : ML & DL METRICS
PART III: COMPREHENSIVE MODEL EVALUATION METRICS
Technical Reference Compendium
Scope: Quantitative Metrics
Format: 3-Page Reference Guide
Evaluation metrics provide quantitative measures of a model's operational performance. Selectin
--------------------------------------------------------------------------------
Rank 2
RRF score : 0.030769
Source    : data\AI Evaluation Report.pdf
Page      : 1
Content   : AI EVALUATION
FRAMEWORKS, BENCHMARKS & SAFETY ASSESSMENTS
Technical Report
Focus: LLM & Frontier Models
Standard: ISO/IEC IEEE 24029
Executive Overview: Robust evaluation of Artiﬁcial Intelligence systems has transitioned from an academic
exercise to
--------------------------------------------------------------------------------
Rank 3
RRF score : 0.030077
Source    : data\AI Evalua

In [1]:
print("Hello")

Hello
